# 15 SARIMA: seasonal differencing, residual diagnostics and the series without the blower message

The daily series shows no weekly periodicity and is stationary by the ADF and KPSS tests, so seasonal differencing is hard to justify. The rule below was fixed before the notebook was run:

1. Specify a model without seasonal differencing (d = 0, D = 0), with orders chosen by AIC within that family. AIC is not compared across differencing orders.
2. Compare it with the seasonally differenced SARIMA(2,0,2)(1,1,1,7) of the originally submitted manuscript ("submitted" in the code) by rolling-origin MASE inside the training period.
3. Score the 30-day blind period once, for both.
4. The undifferenced model becomes primary unless its rolling-origin MASE is more than 5 per cent higher; the other is reported as a sensitivity check.

Because the series is not centred on zero, the undifferenced family is fitted with a constant; the best model of that family without a constant is also reported, so the effect of the constant is visible. Section 6 repeats the comparison on the daily count of the messages other than the blower message.

Input `build/fault_episodes_v4.csv`; outputs `OUTPUT/v4/rev_sarima_*.csv`.

Paper: Sections 2.7.4 and 3.3; Table 11.

In [ ]:
import os, sys, time, warnings, itertools
import numpy as np, pandas as pd
import statsmodels
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.stats.diagnostic import acorr_ljungbox
warnings.filterwarnings("ignore")

A2 = "../ANALYSIS2"
env = os.environ.get
CFG = dict(
    episodes   = os.environ.get("REV_BUILDDIR", "build") + "/fault_episodes_v4.csv",
    ref_series = f"{A2}/OUTPUT/daily_series_v2.csv",
    ref_metrics= f"{A2}/OUTPUT/forecast_metrics.csv",
    outdir     = env("REV_OUTDIR", "OUTPUT") + "/v4",
    blower     = "AUX CAP MACH RM BLOW1 NOT OFF",
    horizon    = 30, m = 7, gap_min_days = 7,
    submitted  = ((2, 0, 2), (1, 1, 1, 7)),
    origin_first = int(env("REV_ORIGIN_FIRST", "120")),   # first rolling origin (training-day index)
    origin_step  = int(env("REV_ORIGIN_STEP", "7")),
    threshold  = 0.05,
)
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| statsmodels", statsmodels.__version__)

## 1. Daily series

The series is rebuilt exactly as in `14_FORECAST`: daily onset counts, with zero-runs of seven days or more masked as out of service. It is compared with the daily series of the submitted manuscript (`daily_series_v2.csv`), from which it differs by design.

In [ ]:
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset"])

def daily(onsets, mask_index=None):
    s = (pd.Series(1, index=pd.DatetimeIndex(onsets)).resample("D").size()
           .asfreq("D").fillna(0).astype(float))
    if mask_index is None:                       # FORECAST rule: zero-runs of >= 7 days
        zero = s.eq(0); block = (zero != zero.shift()).cumsum()
        for _, idx in s.groupby(block).groups.items():
            idx = pd.DatetimeIndex(idx)
            if zero.loc[idx].all() and len(idx) >= CFG["gap_min_days"]:
                s.loc[idx] = np.nan
    else:                                        # reuse the out-of-service days of the full series
        s = s.reindex(mask_index.index).fillna(0)
        s[mask_index.isna()] = np.nan
    return s

s_all = daily(ep.onset)
ref = pd.read_csv(CFG["ref_series"], index_col=0, parse_dates=True).iloc[:, 0]
same = s_all.index.equals(ref.index) and np.allclose(s_all.values, ref.values, equal_nan=True)
print(f"{len(s_all)} days, {int(s_all.isna().sum())} masked; same as the submitted series: {same} (differs by design)")

masked = s_all[s_all.isna()].index
print(f"masked: {masked.min().date()} to {masked.max().date()}")
s_oth = daily(ep.onset[~ep.fault.eq(CFG["blower"])], mask_index=s_all)
print(f"other-label series: mean {s_oth.mean():.2f} per observed day (all labels {s_all.mean():.2f})")

## 2. Functions

In [ ]:
H, M = CFG["horizon"], CFG["m"]

def mase_scale(train):
    v = train.values
    return np.nanmean(np.abs(v[M:] - v[:-M]))

def score(y, pred, train):
    y, pred = np.asarray(y, float), np.asarray(pred, float)
    ok = ~np.isnan(y)
    mae = np.mean(np.abs(y[ok] - pred[ok]))
    rmse = np.sqrt(np.mean((y[ok] - pred[ok]) ** 2))
    return dict(MAE=mae, RMSE=rmse, MASE=mae / mase_scale(train), days=int(ok.sum()))

def naive(train, h):
    return np.repeat(train.dropna().iloc[-1], h)

def seasonal_naive(train, h):
    last = train.ffill().iloc[-M:].values
    return np.array([last[i % M] for i in range(h)])

def fit(train, order, seas, trend=None):
    return SARIMAX(train, order=order, seasonal_order=seas, trend=trend,
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

def select(train, orders, seasonals, trend):
    best = None
    for o, s in itertools.product(orders, seasonals):
        try:
            r = fit(train, o, s, trend)
        except Exception:
            continue
        if np.isfinite(r.aic) and (best is None or r.aic < best[0]):
            best = (r.aic, o, s)
    return best

SEAS_D0 = [(0, 0, 0, 0), (1, 0, 0, M), (0, 0, 1, M), (1, 0, 1, M)]
ORD_D0 = [(p, 0, q) for p in (0, 1, 2) for q in (0, 1, 2)]
ORD_PUB = [(p, d, q) for p in (0, 1, 2) for d in (0, 1) for q in (0, 1, 2)]

## 3. Specifications on the training period

* **D = 1 (submitted)**: SARIMA(2,0,2)(1,1,1,7), no constant, the model of the submitted manuscript.
* **D = 0 with constant**: d = 0 and D = 0, orders by AIC over p, q in {0, 1, 2} and the four undifferenced seasonal forms; all candidates share the same likelihood sample, so AIC is comparable within the family.
* **D = 0 without constant**: the same family without a constant, as in the grid of the submitted manuscript, for comparison only.

In [ ]:
def specs_for(series, label):
    train = series.iloc[:-H]
    t0 = time.time()
    d0c = select(train, ORD_D0, SEAS_D0, "c")
    d0n = select(train, ORD_D0, SEAS_D0, None)
    if label == "all labels":
        d1 = (fit(train, *CFG["submitted"]).aic,) + CFG["submitted"]
    else:   # repeat the submitted D = 1 family on this series
        d1 = select(train, ORD_PUB, [(1, 1, 1, M)], None)
    sp = {"D=1 (submitted)": dict(order=d1[1], seas=d1[2], trend=None, aic=d1[0]),
          "D=0 with constant": dict(order=d0c[1], seas=d0c[2], trend="c", aic=d0c[0]),
          "D=0 without constant": dict(order=d0n[1], seas=d0n[2], trend=None, aic=d0n[0])}
    for k, v in sp.items():
        print(f"  {label:11s} {k:22s} SARIMA{v['order']}{v['seas']}  trend={v['trend']}  AIC {v['aic']:.1f}")
    print(f"  [{time.time() - t0:.0f} s]")
    return sp

SPECS = {"all labels": specs_for(s_all, "all labels")}

## 4. Rolling-origin comparison inside the training period

Origins every seven days from training day 120 to the last origin that leaves a full
30-day window inside the training period. At each origin the three specifications are
refitted with their orders fixed and forecast 30 days ahead; masked days in a window are
not scored. MASE uses the scale of the data up to that origin. The blind period is not
touched.

In [ ]:
def rolling(series, specs, label):
    train = series.iloc[:-H]
    n = len(train)
    origins = list(range(CFG["origin_first"], n - H + 1, CFG["origin_step"]))
    rows = []
    for o in origins:
        tr, te = train.iloc[:o], train.iloc[o:o + H]
        if te.notna().sum() < 7:
            continue
        preds = {"Naive": naive(tr, H), "Seasonal naive": seasonal_naive(tr, H)}
        for k, v in specs.items():
            try:
                preds[k] = fit(tr, v["order"], v["seas"], v["trend"]).forecast(H).values
            except Exception:
                preds[k] = np.full(H, np.nan)
        for k, p in preds.items():
            r = score(te.values, p, tr)
            r.update(series=label, origin=tr.index[-1].date(), model=k)
            rows.append(r)
    out = pd.DataFrame(rows)
    print(f"{label}: {out.origin.nunique()} origins, {out.origin.min()} to {out.origin.max()}")
    return out

t0 = time.time()
roll = rolling(s_all, SPECS["all labels"], "all labels")
summ = roll.groupby("model")[["MASE", "MAE", "RMSE"]].mean().sort_values("MASE")
print(summ.round(3).to_string()); print(f"[{time.time() - t0:.0f} s]")

d1m = summ.loc["D=1 (submitted)", "MASE"]; d0m = summ.loc["D=0 with constant", "MASE"]
excess = d0m / d1m - 1
PRIMARY = "D=0 with constant" if excess <= CFG["threshold"] else "D=1 (submitted)"
print(f"\nD=0 rolling MASE is {100*excess:+.1f}% against D=1 -> primary: {PRIMARY}")

## 5. Blind 30-day period and residual diagnostics

In [ ]:
def blind(series, specs, label):
    train, test = series.iloc[:-H], series.iloc[-H:]
    rows, fits = [], {}
    preds = {"Naive": naive(train, H), "Seasonal naive": seasonal_naive(train, H)}
    for k, v in specs.items():
        fits[k] = fit(train, v["order"], v["seas"], v["trend"])
        preds[k] = fits[k].forecast(H).values
    for k, p in preds.items():
        r = score(test.values, p, train); r.update(series=label, model=k); rows.append(r)
    return pd.DataFrame(rows), fits

def diagnostics(res, spec, label, name):
    burn = spec["order"][1] + spec["seas"][1] * spec["seas"][3]
    z = pd.Series(res.filter_results.standardized_forecasts_error[0], index=res.data.row_labels)
    z = z.iloc[max(burn, 1):].dropna()
    lb = acorr_ljungbox(z, lags=[7, 14], return_df=True)
    r = acf(z, nlags=7, fft=False)
    return dict(series=label, model=name, residuals=len(z),
                LB7_Q=lb.loc[7, "lb_stat"], LB7_p=lb.loc[7, "lb_pvalue"],
                LB14_Q=lb.loc[14, "lb_stat"], LB14_p=lb.loc[14, "lb_pvalue"],
                **{f"acf{k}": r[k] for k in (1, 2, 3, 7)})

bl, FITS = blind(s_all, SPECS["all labels"], "all labels")
bl = bl.set_index("model")
print(bl[["MAE", "RMSE", "MASE"]].round(3).sort_values("MASE").to_string())

pub = pd.read_csv(CFG["ref_metrics"], index_col=0)
rep = abs(bl.loc["D=1 (submitted)", "MASE"] - pub.loc["SARIMA", "MASE"]) < 5e-4
print(f"\nSubmitted SARIMA reproduced (MASE {pub.loc['SARIMA', 'MASE']}): {'OK' if rep else 'DIFFERENT'}")
if False:  # corrected log: comparison with the submitted results only
    assert rep, "submitted SARIMA not reproduced; stop and investigate"

sn = bl.loc["Seasonal naive", "MASE"]
for k in SPECS["all labels"]:
    print(f"  {k:22s} improvement on seasonal naive: {100*(1 - bl.loc[k, 'MASE']/sn):.1f}%")

diag = pd.DataFrame([diagnostics(FITS[k], SPECS["all labels"][k], "all labels", k) for k in FITS])
print("\nResidual diagnostics (standardised one-step residuals, observed days)")
print(diag.round(3).to_string(index=False))

## 6. The series without the blower supervision message

The blower message is absent before 31 January 2025 and then forms more than half of the daily onsets. The same steps are repeated on the daily count of the other messages, with the same out-of-service days masked.

In [ ]:
obs = s_oth.iloc[:-H].dropna()
a, k = adfuller(obs, autolag="AIC"), kpss(obs, regression="c", nlags="auto")
r = acf(s_oth.iloc[:-H], nlags=14, fft=False, missing="conservative")
print(f"other labels, training period: ADF p={a[1]:.4f}  KPSS p={k[1]:.4f}  "
      f"mean {obs.mean():.2f}  variance/mean {obs.var()/obs.mean():.2f}")
print(f"  ACF lag 1 {r[1]:.3f}  lag 2 {r[2]:.3f}  lag 3 {r[3]:.3f}  lag 7 {r[7]:.3f}  lag 14 {r[14]:.3f}")
ra = acf(s_all.iloc[:-H], nlags=14, fft=False, missing="conservative")
print(f"all labels,   for comparison:  ACF lag 1 {ra[1]:.3f}  lag 2 {ra[2]:.3f}  lag 3 {ra[3]:.3f}  lag 7 {ra[7]:.3f}")

SPECS["other labels"] = specs_for(s_oth, "other labels")
roll_o = rolling(s_oth, SPECS["other labels"], "other labels")
summ_o = roll_o.groupby("model")[["MASE", "MAE", "RMSE"]].mean().sort_values("MASE")
print(summ_o.round(3).to_string())
bl_o, FITS_O = blind(s_oth, SPECS["other labels"], "other labels")
print(bl_o.set_index("model")[["MAE", "RMSE", "MASE"]].round(3).sort_values("MASE").to_string())
diag_o = pd.DataFrame([diagnostics(FITS_O[k], SPECS["other labels"][k], "other labels", k) for k in FITS_O])
print(diag_o.round(3).to_string(index=False))

## 7. Save

In [ ]:
spec_rows = [dict(series=lab, model=k, order=str(v["order"]), seasonal=str(v["seas"]), trend=v["trend"], aic=v["aic"]) for lab, sp in SPECS.items() for k, v in sp.items()]
pd.DataFrame(spec_rows).to_csv(f"{CFG['outdir']}/rev_sarima_specs.csv", index=False)
pd.concat([roll, roll_o]).to_csv(f"{CFG['outdir']}/rev_sarima_rolling.csv", index=False)
pd.concat([bl.reset_index(), bl_o]).to_csv(f"{CFG['outdir']}/rev_sarima_blind.csv", index=False)
pd.concat([diag, diag_o]).to_csv(f"{CFG['outdir']}/rev_sarima_diagnostics.csv", index=False)
print(f"Primary specification under the agreed rule: {PRIMARY}")

___
**END**